In [2]:
import osmnx as ox
from scipy.spatial import cKDTree
from sklearn.neighbors import KernelDensity
import numpy as np
import geopandas as gpd
import networkx as nx
import pandas as pd

In [3]:
gdf = pd.read_pickle("crime_preprocessed.pkl")
print(gdf.head())


         DATE OCC  TIME OCC  Crm Cd  \
644982 2023-06-17  11:40:00     442   
644983 2023-02-11  16:30:00     440   
644984 2023-09-19  10:30:00     230   
644985 2023-04-10  08:16:00     331   
644986 2023-12-01  01:10:00     626   

                                              Crm Cd Desc  Part 1-2  \
644982           shoplifting - petty theft ($950 & under)         1   
644983                 theft plain - petty ($950 & under)         1   
644984     assault with deadly weapon, aggravated assault         1   
644985  theft from motor vehicle - grand ($950.01 and ...         1   
644986                  intimate partner - simple assault         2   

          AREA NAME      LAT       LON  is_serious_crime            DATETIME  \
644982    hollywood  34.0981 -118.3092                 1 2023-06-17 11:40:00   
644983      central  34.0396 -118.2726                 1 2023-02-11 16:30:00   
644984  n hollywood  34.1721 -118.3616                 1 2023-09-19 10:30:00   
644985      centra

In [4]:
print(gdf.columns)

Index(['DATE OCC', 'TIME OCC', 'Crm Cd', 'Crm Cd Desc', 'Part 1-2',
       'AREA NAME', 'LAT', 'LON', 'is_serious_crime', 'DATETIME', 'HOUR',
       'DAY_OF_WEEK', 'MONTH', 'IS_WEEKEND', 'hour_sin', 'hour_cos',
       'crime_category', 'geometry', 'x', 'y', 'severity'],
      dtype='object')


In [5]:
place_name = "Los Angeles, California, USA"

G = ox.graph_from_place(place_name, network_type="drive")

In [6]:
G_proj = ox.project_graph(G, to_crs="EPSG:32611")

In [7]:
edges = ox.graph_to_gdfs(G_proj, nodes=False)
edges = edges.reset_index()

In [8]:
edges["midpoint"] = edges.geometry.interpolate(0.5, normalized=True)

edges["x"] = edges["midpoint"].x
edges["y"] = edges["midpoint"].y

In [9]:


crime_coords = np.column_stack((gdf["x"], gdf["y"]))

kde = KernelDensity(
    bandwidth=200,
    kernel="gaussian",
    metric="euclidean"
)

kde.fit(crime_coords)

edge_coords = np.column_stack((edges["x"], edges["y"]))

log_density = kde.score_samples(edge_coords)

edges["crime_kde"] = np.exp(log_density)

edges["crime_kde_norm"] = edges["crime_kde"] / edges["crime_kde"].max()

In [10]:
crime_points = np.vstack([gdf["x"], gdf["y"]]).T
crime_tree = cKDTree(crime_points)

In [11]:
edge_points = np.column_stack((edges["x"], edges["y"]))

radius = 200

neighbors = crime_tree.query_ball_point(edge_points, r=radius)

edges["local_crime_count"] = [len(n) for n in neighbors]

In [12]:
# ── Step 1a: compute per-hour crime counts per edge ──────────────────────────

crime_hours = gdf["HOUR"].values  # shape: (n_crimes,)
crime_severity = gdf["severity"].values  # shape: (n_crimes,)

# For each edge, count how many crimes happened at each hour of the day
hourly_crime_counts = []

for idx_list in neighbors:
    if len(idx_list) == 0:
        hourly_crime_counts.append({h: 0 for h in range(24)})
    else:
        hours_in_radius = crime_hours[list(idx_list)]
        unique_hours, counts = np.unique(hours_in_radius, return_counts=True)
        hour_dict = {h: 0 for h in range(24)}
        for h, c in zip(unique_hours, counts):
            hour_dict[h] = c
        hourly_crime_counts.append(hour_dict)

hourly_df = pd.DataFrame(hourly_crime_counts)  # shape: (n_edges, 24)
hourly_df.columns = [f"crime_hour_{h}" for h in range(24)]
hourly_df.index = edges.index  # align index

edges = pd.concat([edges, hourly_df], axis=1)

print("Hourly crime columns added:", hourly_df.shape)
print("Sample — edge 0 hourly counts:")
print(edges[[f"crime_hour_{h}" for h in range(24)]].iloc[0].to_dict())

Hourly crime columns added: (136103, 24)
Sample — edge 0 hourly counts:
{'crime_hour_0': 2, 'crime_hour_1': 5, 'crime_hour_2': 2, 'crime_hour_3': 2, 'crime_hour_4': 4, 'crime_hour_5': 6, 'crime_hour_6': 2, 'crime_hour_7': 1, 'crime_hour_8': 5, 'crime_hour_9': 4, 'crime_hour_10': 7, 'crime_hour_11': 3, 'crime_hour_12': 15, 'crime_hour_13': 10, 'crime_hour_14': 7, 'crime_hour_15': 9, 'crime_hour_16': 9, 'crime_hour_17': 9, 'crime_hour_18': 10, 'crime_hour_19': 6, 'crime_hour_20': 10, 'crime_hour_21': 8, 'crime_hour_22': 4, 'crime_hour_23': 9}


In [13]:
severity_values = gdf["severity"].values

edges["local_severity"] = [
    severity_values[idx].mean() if len(idx) > 0 else 0
    for idx in neighbors
]

In [14]:
edges["local_crime_norm"] = edges["local_crime_count"] / edges["local_crime_count"].max()

edges["local_severity_norm"] = edges["local_severity"] / edges["local_severity"].max()

In [21]:
# ── Add weekend_crime_fraction ────────────────────────────────────────────────

crime_dow = gdf["DAY_OF_WEEK"].values

weekend_fractions = []
for idx_list in neighbors:
    if len(idx_list) == 0:
        weekend_fractions.append(0.286)  # base rate 2/7 days
    else:
        dows = crime_dow[list(idx_list)]
        weekend_fractions.append(float(np.mean(np.isin(dows, [5, 6]))))

edges["weekend_crime_fraction"] = weekend_fractions

In [15]:
crime_xy = np.column_stack((gdf["x"], gdf["y"]))
severity_vals = gdf["severity"].values

weighted_scores = []

for edge_idx, crime_idx_list in enumerate(neighbors):

    if len(crime_idx_list) == 0:
        weighted_scores.append(0)
        continue

    edge_x = edges.iloc[edge_idx]["x"]
    edge_y = edges.iloc[edge_idx]["y"]

    crime_coords = crime_xy[crime_idx_list]

    distances = np.sqrt(
        (crime_coords[:,0] - edge_x)**2 +
        (crime_coords[:,1] - edge_y)**2
    )

    weights = 1 / (distances + 1)

    weighted_score = np.sum(weights * severity_vals[crime_idx_list])

    weighted_scores.append(weighted_score)

edges["distance_weighted_crime"] = weighted_scores

In [16]:
edges["distance_weighted_crime_norm"] = (
    edges["distance_weighted_crime"] /
    edges["distance_weighted_crime"].max()
)

In [17]:
degree_dict = dict(G_proj.degree())

edges["degree_u"] = edges["u"].map(degree_dict)
edges["degree_v"] = edges["v"].map(degree_dict)

edges["avg_degree"] = (edges["degree_u"] + edges["degree_v"]) / 2

In [18]:
bc = nx.betweenness_centrality(G_proj, k=1500, seed=42)

edges["bc_u"] = edges["u"].map(bc)
edges["bc_v"] = edges["v"].map(bc)

edges["edge_betweenness"] = (edges["bc_u"] + edges["bc_v"]) / 2

In [19]:
edges["highway"] = edges["highway"].astype(str)

In [22]:
hour_cols = [f"crime_hour_{h}" for h in range(24)]

edges_ml = edges[
    [
        "u",
        "v",
        "length",
        "highway",
        "local_crime_norm",
        "local_severity_norm",
        "distance_weighted_crime_norm",
        "crime_kde_norm",
        "avg_degree",
        "edge_betweenness",
        "weekend_crime_fraction",   # ← new
        "x",
        "y",
    ] + hour_cols
]

edges_ml.to_pickle("edge_features.pkl")
print("Saved edge_features.pkl with shape:", edges_ml.shape)

Saved edge_features.pkl with shape: (136103, 37)


In [19]:
edges_ml.to_pickle("edge_features.pkl")

In [20]:
import osmnx as ox

ox.save_graphml(G_proj, "los_angeles_graph.graphml")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

# Get edge KDE values in graph order
edge_colors = []
for u, v, k, data in G_proj.edges(keys=True, data=True):
    # match to edges dataframe
    mask = (edges_ml["u"] == u) & (edges_ml["v"] == v)
    if mask.any():
        val = edges_ml.loc[mask, "crime_kde_norm"].values[0]
    else:
        val = 0
    edge_colors.append(val)

# Normalize for colormap
norm = mcolors.Normalize(vmin=0, vmax=1)
cmap = cm.YlOrRd
ec = [cmap(norm(v)) for v in edge_colors]

# Plot
fig, ax = ox.plot_graph(
    G_proj,
    edge_color=ec,
    edge_linewidth=0.6,
    edge_alpha=0.8,
    node_size=0,
    bgcolor="black",
    show=False,
    close=False
)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label="Crime KDE density (normalised)", shrink=0.5)

ax.set_title("Crime density heatmap — Los Angeles road network", color="white", fontsize=13)

plt.savefig("crime_kde_heatmap.png", dpi=150, bbox_inches="tight", facecolor="black")
plt.show()
print("Saved crime_kde_heatmap.png")